# Klasifikasi Tingkat Pendapatan Individu Berdasarkan Data Sensus Menggunakan Decision Tree

**TA-05 Praktikum Pembelajaran Mesin** | Dataset: *Adult Census Income* (Kaggle, uciml)

Alur: EDA, pra-pemrosesan, **Model A (tanpa batas, berpotensi overfit)** dan **Model B (di-pruning)**, perbandingan keduanya, visualisasi dan interpretasi pohon, lalu penyimpanan model untuk aplikasi Flask.

In [ ]:
import os
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, roc_auc_score

from tree_utils import (NUM_COLS, CAT_COLS, FEATURES, CLASS_NAMES,
                        extract_leaf_rules, importance_by_variable, explain_path)

sns.set_theme(style="whitegrid")
RANDOM_STATE = 42
B_DEPTH = 4          # kedalaman maksimum Model B (pruned)

os.makedirs("gambar", exist_ok=True)
os.makedirs("model", exist_ok=True)

## 1. Memuat Data

In [ ]:
df = pd.read_csv("data/adult.csv")
print("Ukuran data:", df.shape)
df.head()

## 2. Pembersihan Data

Pada dataset ini nilai yang hilang ditulis sebagai tanda tanya (`?`), bukan `NaN`, sehingga harus diubah dulu agar terdeteksi sebagai data hilang.

In [ ]:
df = df.replace("?", np.nan)

missing = df.isna().sum()
print("Nilai hilang per kolom:")
print(missing[missing > 0])

n_awal = len(df)
df = df.drop_duplicates().reset_index(drop=True)
print("\nBaris duplikat dibuang:", n_awal - len(df), "| sisa baris:", len(df))

**Pemilihan fitur.** Model memakai 8 fitur: `age`, `education.num`, `capital.gain`, `capital.loss`, `hours.per.week`, `workclass`, `marital.status`, dan `occupation`. Kolom yang tidak dipakai:
- `fnlwgt`: bobot sampling sensus, bukan karakteristik individu.
- `education`: duplikat dari `education.num` (versi angkanya).
- `relationship`: sangat terkait dengan `marital.status` dan jenis kelamin.
- `race`, `sex`, `native.country`: atribut sensitif atau hampir seragam (sebagian besar warga AS). Memakai ras dan jenis kelamin untuk menilai pendapatan seseorang tidak pantas untuk alat pendukung keputusan.

In [ ]:
y = (df["income"] == ">50K").astype(int)       # 1 = pendapatan > 50K
X = df[FEATURES].copy()
print("Fitur numerik   :", NUM_COLS)
print("Fitur kategorikal:", CAT_COLS)
print("Proporsi kelas >50K:", round(y.mean(), 4))

## 3. Exploratory Data Analysis (EDA)

In [ ]:
plt.figure(figsize=(6, 4))
ax = sns.countplot(x="income", data=df)
for p in ax.patches:
    ax.annotate(int(p.get_height()), (p.get_x() + p.get_width() / 2, p.get_height()), ha="center", va="bottom")
plt.title("Distribusi Kelas Pendapatan")
plt.xlabel("Kelas pendapatan")
plt.ylabel("Jumlah")
plt.tight_layout()
plt.savefig("gambar/01_distribusi_target.png", dpi=150)
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col in zip(axes, ["age", "education.num", "hours.per.week"]):
    sns.histplot(data=df, x=col, hue="income", stat="density", common_norm=False, element="step", ax=ax)
    ax.set_title(f"Distribusi {col} menurut kelas")
plt.tight_layout()
plt.savefig("gambar/02_distribusi_numerik.png", dpi=150)
plt.show()

print("Persentase yang memiliki capital.gain > 0:")
print((df["capital.gain"].gt(0).groupby(df["income"]).mean() * 100).round(1))

In [ ]:
tmp = X.copy()
tmp["tinggi"] = y
tmp[CAT_COLS] = tmp[CAT_COLS].fillna("Unknown")

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, col in zip(axes, CAT_COLS):
    rate = (tmp.groupby(col)["tinggi"].mean() * 100).sort_values()
    ax.barh(rate.index, rate.values)
    ax.set_title(f"% berpendapatan >50K menurut {col}")
    ax.set_xlabel("% >50K")
plt.tight_layout()
plt.savefig("gambar/03_kategori_vs_target.png", dpi=150)
plt.show()

## 4. Pembagian Data dan Pra-pemrosesan

Data dibagi **70% latih dan 30% uji** secara berstrata (proporsi kelas terjaga). Pra-pemrosesan dibungkus dalam *pipeline* supaya aplikasi web memperlakukan input persis seperti saat pelatihan:
- Fitur kategorikal: nilai hilang diisi kategori `Unknown`, lalu di-*one-hot encode*. Pada kolom seperti `workclass` dan `occupation`, tanda `?` sering berarti tidak bekerja atau tidak diketahui, jadi dijadikan kategori sendiri lebih tepat daripada diisi modus.
- Fitur numerik: dipakai apa adanya (Decision Tree tidak memerlukan *scaling*).

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=RANDOM_STATE, stratify=y
)
print("Latih:", X_train.shape, "| Uji:", X_test.shape)


def make_pipeline(**tree_params):
    prep = ColumnTransformer([
        ("num", "passthrough", NUM_COLS),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="constant", fill_value="Unknown")),
            ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
        ]), CAT_COLS),
    ])
    return Pipeline([("prep", prep), ("tree", DecisionTreeClassifier(random_state=RANDOM_STATE, **tree_params))])

## 5. Model A (Tanpa Batas) dan Model B (Di-pruning)

In [ ]:
# Model A: dibiarkan tumbuh bebas (max_depth=None)
model_a = make_pipeline().fit(X_train, y_train)

# Model B: dibatasi kedalamannya (pre-pruning)
model_b = make_pipeline(max_depth=B_DEPTH).fit(X_train, y_train)

for nama, m in [("Model A (tanpa batas)", model_a), ("Model B (max_depth=%d)" % B_DEPTH, model_b)]:
    t = m.named_steps["tree"]
    print(f"{nama}: kedalaman={t.get_depth()}, jumlah daun={t.get_n_leaves()}, "
          f"akurasi latih={m.score(X_train, y_train):.4f}, akurasi uji={m.score(X_test, y_test):.4f}")

### 5.1 Mencari batas kedalaman: akurasi latih vs validasi silang vs uji

In [ ]:
rows = []
for d in range(1, 21):
    m = make_pipeline(max_depth=d)
    cv = cross_val_score(m, X_train, y_train, cv=5).mean()      # validasi silang hanya pada data latih
    m.fit(X_train, y_train)
    rows.append({"max_depth": d, "akurasi_latih": m.score(X_train, y_train),
                 "akurasi_cv": cv, "akurasi_uji": m.score(X_test, y_test)})
sweep = pd.DataFrame(rows)

best = sweep.loc[sweep["akurasi_cv"].idxmax()]
print("Kedalaman terbaik menurut validasi silang:", int(best["max_depth"]), "| akurasi CV:", round(best["akurasi_cv"], 4))

x_pos = list(sweep["max_depth"]) + [21]
lat = list(sweep["akurasi_latih"]) + [model_a.score(X_train, y_train)]
uji = list(sweep["akurasi_uji"]) + [model_a.score(X_test, y_test)]
cvs = list(sweep["akurasi_cv"]) + [np.nan]

plt.figure(figsize=(9, 5))
plt.plot(x_pos, lat, marker="o", label="Akurasi latih")
plt.plot(x_pos, cvs, marker="s", label="Akurasi validasi silang")
plt.plot(x_pos, uji, marker="^", label="Akurasi uji")
plt.xticks(x_pos, [str(d) for d in range(1, 21)] + ["None"])
plt.axvline(B_DEPTH, color="gray", linestyle=":")
plt.title("Akurasi menurut kedalaman maksimum pohon")
plt.xlabel("max_depth")
plt.ylabel("Akurasi")
plt.legend()
plt.tight_layout()
plt.savefig("gambar/04_akurasi_vs_kedalaman.png", dpi=150)
plt.show()
sweep.round(4)

## 6. Evaluasi dan Perbandingan Model

In [ ]:
def evaluasi(nama, m):
    pred = m.predict(X_test)
    proba = m.predict_proba(X_test)[:, 1]
    print("=" * 60)
    print(nama)
    print("--- Confusion Matrix ---")
    print(confusion_matrix(y_test, pred))
    print("\n--- Classification Report ---")
    print(classification_report(y_test, pred, target_names=CLASS_NAMES))
    return pred, proba

pred_a, proba_a = evaluasi("MODEL A (tanpa batas)", model_a)
pred_b, proba_b = evaluasi("MODEL B (di-pruning, max_depth=%d)" % B_DEPTH, model_b)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
for ax, (nama, pred) in zip(axes, [("Model A (tanpa batas)", pred_a), ("Model B (max_depth=%d)" % B_DEPTH, pred_b)]):
    sns.heatmap(confusion_matrix(y_test, pred), annot=True, fmt="d", cmap="Blues",
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax)
    ax.set_title(nama)
    ax.set_xlabel("Label prediksi")
    ax.set_ylabel("Label sebenarnya")
plt.tight_layout()
plt.savefig("gambar/05_confusion_matrix.png", dpi=150)
plt.show()

In [ ]:
ringkasan = pd.DataFrame({
    "Model": ["A (tanpa batas)", "B (max_depth=%d)" % B_DEPTH],
    "Kedalaman": [model_a.named_steps["tree"].get_depth(), model_b.named_steps["tree"].get_depth()],
    "Jumlah daun": [model_a.named_steps["tree"].get_n_leaves(), model_b.named_steps["tree"].get_n_leaves()],
    "Akurasi latih": [model_a.score(X_train, y_train), model_b.score(X_train, y_train)],
    "Akurasi uji": [model_a.score(X_test, y_test), model_b.score(X_test, y_test)],
    "AUC uji": [roc_auc_score(y_test, proba_a), roc_auc_score(y_test, proba_b)],
})
ringkasan["Selisih latih-uji"] = ringkasan["Akurasi latih"] - ringkasan["Akurasi uji"]
ringkasan.round(4)

In [ ]:
plt.figure(figsize=(6.5, 5))
for nama, proba in [("Model A (tanpa batas)", proba_a), ("Model B (max_depth=%d)" % B_DEPTH, proba_b)]:
    fpr, tpr, _ = roc_curve(y_test, proba)
    plt.plot(fpr, tpr, label=f"{nama} (AUC = {roc_auc_score(y_test, proba):.2f})")
plt.plot([0, 1], [0, 1], "k--", label="Tebakan acak")
plt.xlabel("False Positive Rate (FPR)")
plt.ylabel("True Positive Rate (TPR)")
plt.title("Kurva ROC")
plt.legend()
plt.tight_layout()
plt.savefig("gambar/06_roc.png", dpi=150)
plt.show()

## 7. Visualisasi dan Interpretasi Model B

In [ ]:
prep_b = model_b.named_steps["prep"]
tree_b = model_b.named_steps["tree"]
nama_fitur = [n.replace("num__", "").replace("cat__", "") for n in prep_b.get_feature_names_out()]

plt.figure(figsize=(26, 11))
plot_tree(tree_b, feature_names=nama_fitur, class_names=CLASS_NAMES,
          filled=True, rounded=True, fontsize=8)
plt.title("Visualisasi Pohon Keputusan Model B (max_depth=%d)" % B_DEPTH)
plt.tight_layout()
plt.savefig("gambar/07_pohon_model_b.png", dpi=150)
plt.show()

### Aturan "jika-maka"
Tiga jalur dari akar ke daun dengan sampel terbanyak diterjemahkan menjadi aturan. *Kemurnian* adalah persentase sampel di daun yang termasuk kelas prediksinya.

In [ ]:
def tampilkan(rule):
    print("JIKA " + "\n  DAN ".join(rule["syarat"]))
    print(f"MAKA pendapatan {rule['kelas']}  (sampel={rule['sampel']}, kemurnian={rule['kemurnian']:.1%})\n")

print("### 3 daun terbesar ###\n")
for r in extract_leaf_rules(model_b, top_n=3):
    tampilkan(r)

print("### 2 daun terbesar yang memprediksi >50K ###\n")
for r in extract_leaf_rules(model_b, top_n=2, target_class=1):
    tampilkan(r)

### Feature Importance

In [ ]:
imp_var = importance_by_variable(model_b)
imp_df = pd.DataFrame({"Variabel": list(imp_var.keys()), "Pentingnya": list(imp_var.values())})

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.barplot(x="Pentingnya", y="Variabel", data=imp_df, ax=axes[0], color="steelblue")
axes[0].set_title("Pentingnya per variabel asli (one-hot dijumlahkan)")

imp_fitur = pd.DataFrame({"Fitur": nama_fitur, "Pentingnya": tree_b.feature_importances_}).sort_values("Pentingnya", ascending=False).head(8)
sns.barplot(x="Pentingnya", y="Fitur", data=imp_fitur, ax=axes[1], color="steelblue")
axes[1].set_title("8 fitur terpenting (setelah one-hot)")
plt.tight_layout()
plt.savefig("gambar/08_feature_importance.png", dpi=150)
plt.show()
imp_df.round(4)

## 8. Menyimpan Model dengan Joblib
Model B dipilih untuk aplikasi karena lebih sederhana, mudah dijelaskan, dan jauh lebih kecil kesenjangannya antara data latih dan uji. File `decision_tree_pruned.joblib` memuat seluruh *pipeline* (pra-pemrosesan + pohon).

In [ ]:
joblib.dump(model_b, "model/decision_tree_pruned.joblib")

categories = {c: sorted(X[c].dropna().unique().tolist()) + ["Unknown"] for c in ["workclass", "occupation"]}
categories["marital.status"] = sorted(X["marital.status"].dropna().unique().tolist())

config = {
    "features": FEATURES,
    "b_depth": B_DEPTH,
    "categories": categories,
    "education_map": {int(k): v for k, v in df.groupby("education.num")["education"].first().to_dict().items()},
    "ranges": {c: [float(X[c].min()), float(X[c].max())] for c in NUM_COLS},
    "random_state": RANDOM_STATE,
}
with open("model/config.json", "w", encoding="utf-8") as f:
    json.dump(config, f, indent=2, ensure_ascii=False)

metrics = {
    "model_a": {"akurasi_latih": round(model_a.score(X_train, y_train), 4), "akurasi_uji": round(model_a.score(X_test, y_test), 4),
                "auc": round(roc_auc_score(y_test, proba_a), 4), "daun": int(model_a.named_steps["tree"].get_n_leaves())},
    "model_b": {"akurasi_latih": round(model_b.score(X_train, y_train), 4), "akurasi_uji": round(model_b.score(X_test, y_test), 4),
                "auc": round(roc_auc_score(y_test, proba_b), 4), "daun": int(model_b.named_steps["tree"].get_n_leaves())},
    "importance": {k: round(v, 4) for k, v in imp_var.items()},
    "n_latih": int(len(X_train)), "n_uji": int(len(X_test)),
}
with open("model/metrics.json", "w", encoding="utf-8") as f:
    json.dump(metrics, f, indent=2, ensure_ascii=False)

print(sorted(os.listdir("model")))

In [ ]:
# Uji cepat: model tersimpan dapat dimuat dan menjelaskan jalur keputusannya
loaded = joblib.load("model/decision_tree_pruned.joblib")
contoh = X_test.iloc[[0]]
info = explain_path(loaded, contoh)
print("Data contoh:", contoh.iloc[0].to_dict())
print("Prediksi:", CLASS_NAMES[info["prediction"]], "| peluang >50K:", round(info["p_high"], 3))
print("Jalur keputusan:")
for s in info["steps"]:
    print("  -", s)

## 9. Catatan untuk Laporan
Setelah **Run All**, ambil dari output di atas:
- Jumlah baris, nilai hilang, dan baris duplikat (bagian 2)
- Distribusi target dan temuan EDA (bagian 3)
- Perbandingan akurasi latih vs uji Model A dan B, serta grafik kedalaman (bagian 5 dan 6)
- Confusion matrix dan classification report kedua model (bagian 6)
- Pohon Model B, tiga aturan jika-maka, dan feature importance (bagian 7)